In [ ]:
# Navigating the City: A MuZero-Based Delivery Bot Simulation
# This code simulates a delivery bot using a simplified MuZero architecture to make decisions based on its current state and potential actions. 
# The bot processes its sensor data to predict the best move 
import torch
import torch.nn as nn
# Step 2: Defining the MuZero-Style Network structure consisting of Representation, Dynamics, and Prediction components.
class MuZeroDeliveryNet(nn.Module):
    def __init__(self, state_size=5, action_size=4, hidden_size=64):
        super(MuZeroDeliveryNet, self).__init__()
        # Representation: Raw data -> internal summary
        self.representation = nn.Sequential(nn.Linear(state_size, hidden_size), nn.ReLU())
        # Dynamics: Internal summary + Action -> predicted next summary
        self.dynamics = nn.Sequential(nn.Linear(hidden_size + action_size, hidden_size), nn.ReLU())
        # Prediction Heads: Resulting Reward, Policy, and Value
        self.reward_head = nn.Linear(hidden_size, 1)
        self.policy_head = nn.Linear(hidden_size, action_size)
        self.value_head = nn.Linear(hidden_size, 1)
 # Step 3: Initial Observation - The bot "opens its eyes" 
    def initial_inference(self, raw_state):
        h = self.representation(raw_state)
        return h, self.policy_head(h), self.value_head(h)
# Step 5: The What If
    def recurrent_inference(self, h, action):
        next_h = self.dynamics(torch.cat([h, action], dim=1))
        return next_h, self.reward_head(next_h), self.policy_head(next_h), self.value_head(next_h)

# --- OUTPUT (the environment) ---
# Inputs: [X_Pos, Y_Pos, Traffic_Density, Battery_Life, Urgency]
current_sensors = torch.tensor([[12.0, 45.0, 0.7, 0.85, 1.0]]) 
model = MuZeroDeliveryNet()

# Step 3(continued): Initial Observation (The Bot 'sees' its current spot)
h_state, policy, value = model.initial_inference(current_sensors)
print(f"--- STEP 1: INITIAL OBSERVATION ---")
print(f"Bot's initial confidence in success (Value): {value.item():.4f}")

# Step 4: Recurrent Step (The Bot 'imagines' turning LEFT)
# Action Vector: [Up=0, Down=0, Left=1, Right=0]
action_left = torch.tensor([[0, 0, 1.0, 0]]) 

# Step 6:Predicting the FutureThe Dynamics Function calculates the 'imagined' result
next_h, reward, next_p, next_v = model.recurrent_inference(h_state, action_left)

print(f"\n--- STEP 2: IMAGINED FUTURE (Turning Left) ---")
print(f"Predicted time saved if turning left (Reward): {reward.item():.4f}")
print(f"Predicted overall success for this route (Future Value): {next_v.item():.4f}")

--- STEP 1: INITIAL OBSERVATION ---
Bot's initial confidence in success (Value): -0.5185

--- STEP 2: IMAGINED FUTURE (Turning Left) ---
Predicted time saved if turning left (Reward): 0.5212
Predicted overall success for this route (Future Value): -0.6172
